<a href="https://colab.research.google.com/github/Innovatewithapple/RAGPractice/blob/main/InitialRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q datasets transformers

In [45]:
import pandas as pd
from datasets import load_dataset
from torch.utils.data import DataLoader,Dataset
import torch.nn as nn
from transformers import AutoModel,AutoTokenizer,AutoModelForCausalLM
import torch
from tqdm import tqdm
import torch.nn.functional as F

Step1:

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

In [4]:
raw_hf_dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train[:1000]")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Bitext_Sample_Customer_Support_Training_(…):   0%|          | 0.00/19.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

In [5]:
raw_hf_dataset

Dataset({
    features: ['flags', 'instruction', 'category', 'intent', 'response'],
    num_rows: 1000
})

In [6]:
df = pd.DataFrame(raw_hf_dataset)
df

,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...
...,...,...,...,...,...
995,BIL,how to cancel purchase {{Order Number}}?,ORDER,cancel_order,I've decoded that you're looking for assistanc...
996,BLZ,I want help cancellkng purchase {{Order Number}},ORDER,cancel_order,I realized that you're seeking assistance with...
997,BL,I want assistance cancelling purchase {{Order ...,ORDER,cancel_order,I understand your need for assistance in cance...
998,BMW,I try to change several bloody items of order ...,ORDER,change_order,We understand that you would like to make chan...


In [7]:
df['document_text'] = "User Question: " + df['instruction'].astype(str) + " | Protocol Answer: " + df['response'].astype(str)

In [8]:
df = df[['document_text']]

In [9]:
print(df['document_text'].iloc[0])

User Question: question about cancelling order {{Order Number}} | Protocol Answer: I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.


In [10]:
autoToken = AutoTokenizer.from_pretrained('bert-base-uncased')

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
knowledgeText = df['document_text'].tolist()

In [12]:
max_len = 128

In [14]:
print(knowledgeText[0])

User Question: question about cancelling order {{Order Number}} | Protocol Answer: I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.


Step2:

In [15]:
train_tokeniser = autoToken(text=knowledgeText,padding='max_length',max_length=max_len,add_special_tokens=True,truncation=True,return_attention_mask=True,return_tensors='pt')

In [16]:
class RAGKnowledgeDataset(Dataset):
  def __init__(self,encoding):
    self.encoding = encoding

  def __len__(self):
    return len(self.encoding['input_ids'])

  def __getitem__(self,idx):
    input_ids = self.encoding['input_ids'][idx]
    attention_mask = self.encoding['attention_mask'][idx]

    return {
        'input_ids':input_ids,
        'attention_mask':attention_mask
    }

In [17]:
train_ds = RAGKnowledgeDataset(train_tokeniser)
train_loader = DataLoader(dataset=train_ds,batch_size=32,shuffle=False,pin_memory=True,num_workers=2)

We use custom BERTWritter class We remove the random self.outputlayer linear layer and the dropout node We do this because we only want the pure, untampered mathematical sentence representation from Smart Mean shortcut formula

In [18]:
class BERTWritter(nn.Module):
  def __init__(self) -> None:
    super().__init__()
    self.bert = AutoModel.from_pretrained('bert-base-uncased')

  def forward(self,input_ids,attention_mask):
    output = self.bert(input_ids=input_ids,attention_mask=attention_mask)
    x = output.last_hidden_state

    mean = attention_mask.unsqueeze(-1).float()
    mean = (x*mean).sum(dim=1) / mean.sum(dim=1)
    return mean

In [19]:
model_encoder = BERTWritter().to(device)
model_encoder.eval() # Hard rule: Turn off dropout for inference safety!

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTWritter(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affin

In [20]:
#The vector complicaiton loop
database_vectors = []

with torch.no_grad():
  with torch.amp.autocast('cuda'):

    for batch in tqdm(train_loader,desc='Encoding Batchs'):
      input_ids = batch['input_ids'].to(device)
      attention_mask = batch['attention_mask'].to(device)

      # Extract the 768-dimension sentence vector features
      batch_vectors = model_encoder(input_ids, attention_mask)

      # Move to CPU to keep your active GPU VRAM bars clear
      database_vectors.append(batch_vectors.cpu())

# 7. Stack the batch arrays vertically into your permanent Vector Database file
vector_database = torch.cat(database_vectors,dim=0)
print(f"Final Tensor Database Shape: {vector_database.shape}")


/tmp/ipykernel_8568/4118835213.py:5: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with torch.amp.autocast('cuda'):
Encoding Batchs:   0%|          | 0/32 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Encoding Batchs: 100%|██████████| 32/32 [07:25<00:00, 13.92s/it]

Final Tensor Database Shape: torch.Size([1000, 768])


Step3: INITIALIZING THE SEARCH RETRIEVAL ENGINE ---

In [21]:
user_question = 'Hey, I need help canceling the purchase order I just made'

In [22]:
query_tokenized = autoToken(text=user_question,padding='max_length',max_length=max_len,add_special_tokens=True,return_attention_mask=True,return_tensors='pt').to(device)

In [23]:
with torch.no_grad():
  with torch.amp.autocast('cuda'):
    # Output shape is a 1D row vector: [1, 768]
    query_vector = model_encoder(query_tokenized['input_ids'],query_tokenized['attention_mask'])

# 4. THE PRO MIDDLE GLUE: LIGHTNING-FAST COSINE SIMILARITY RETRIEVAL LINE [prev]
# We move the query vector back to CPU to match our vector_database storage tensor location [prev]
query_vector_cpu = query_vector.cpu()

/tmp/ipykernel_8568/2435430604.py:2: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with torch.amp.autocast('cuda'):


In [48]:
# Matrix Multiplication: Multiply your [1, 768] query by the transposed [768, 1000] database
# 1. Manually normalize your query vector to length 1.0 along its 768 features axis
normalized_query = F.normalize(query_vector_cpu, p=2, dim=-1)

# 2. Manually normalize your entire 1,000-row database tensor to length 1.0 along the 768 axis
normalized_db = F.normalize(vector_database, p=2, dim=-1)

# 3. Now, running the pure '@' dot product outputs the exact same clean 0.00 to 1.00 decimal!
similarity_scores = normalized_query @ normalized_db.T

# Extract your summary values (Slicing index 0 because matrix multiplication preserves a 2D batch dimension)
best_score = torch.max(similarity_scores).item()
best_index = torch.argmax(similarity_scores).item()

print(f"Highest Document Match Confidence Score: {best_score:.4f}")

# 🚨 Execute your guardrail filter checks on the clean decimal
if best_score < 0.70:
    fetched_context_document = "I'm sorry, I cannot find any verified corporate protocols in my database to safely answer your question."
else:
    fetched_context_document = knowledgeText[best_index]

Highest Document Match Confidence Score: 0.7346


In [49]:
print("\n--- CONFIRMED CONTEXT SENT TO GPT-2 ---")
print(fetched_context_document)


--- CONFIRMED CONTEXT SENT TO GPT-2 ---
User Question: question about canceling a bloody purchase | Protocol Answer: For sure! I understand that you have some concerns regarding canceling your purchase. I'm here to help. Can you please provide me with the order number of your purchase so that I can assist you more effectively?


In [26]:
print(f"User Query: '{user_question}'")
print(f"Best Matching Document Row Found at Index: {best_index}")
print(f"Confidence Similarity Score: {similarity_scores[0, best_match_index].item():.4f}")
print("\n--- FETCHED REFERENCE CONTEXT PARAGRAPH ---")
print(fetched_context_document)

User Query: 'Hey, I need help canceling the purchase order I just made'
Best Matching Document Row Found at Index: 579
Confidence Similarity Score: 66.6497

--- FETCHED REFERENCE CONTEXT PARAGRAPH ---
User Question: question about canceling a bloody purchase | Protocol Answer: For sure! I understand that you have some concerns regarding canceling your purchase. I'm here to help. Can you please provide me with the order number of your purchase so that I can assist you more effectively?


--- STEP 4: INITIALIZING THE GENERATIVE WRITER ---

In [27]:
class SentimentDataset(Dataset):
    def __init__(self, encoding):
        self.encoding = encoding

    def __len__(self):
        # Returns the clean count of rows in the encoding batch [prev]
        return len(self.encoding['input_ids'])

    def __getitem__(self, idx):
        # Extract the full 1D token and mask vectors for the specific row index
        ids = self.encoding['input_ids'][idx]
        mask = self.encoding['attention_mask'][idx]

        # We return the exact dictionary structures required by the model
        return {
            'input_ids': ids,
            'attention_mask': mask
        }

In [28]:
class CustomGPT2Writer(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # 1. Pull the official pre-trained foundation brain [prev]
        self.gpt2 = AutoModel.from_pretrained('gpt2')

        # 2. Add your custom regularization and classification layers [prev]
        self.dropout = nn.Dropout(0.2)
        self.output_layer = nn.Linear(768, vocab_size)

    def forward(self, input_ids, attention_mask):
        # Pass BOTH parameters down to satisfy the foundation network layers [prev]
        outputs = self.gpt2(input_ids=input_ids, attention_mask=attention_mask)

        # Extract the hidden states matrix (semantic feature vectors) [prev]
        x = outputs.last_hidden_state # Shape: [Batch, Seq_Len, 768]

        # Apply your custom processing nodes
        x = self.dropout(x)
        logits = self.output_layer(x) # Shape out: [Batch, Seq_Len, Vocab_Size]

        return logits

In [29]:
decoder_tokenizer = AutoTokenizer.from_pretrained('gpt2')
decoder_tokenizer.pad_token = decoder_tokenizer.eos_token

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [30]:
decoder_vocab_size = decoder_tokenizer.vocab_size

In [38]:
# decoder_brain = CustomGPT2Writer(vocab_size=decoder_tokenizer.vocab_size).to(device)
# decoder_brain.eval()
# Use the ForCausalLM variant—it has its output layers fully pre-trained by OpenAI!
decoder_brain = AutoModelForCausalLM.from_pretrained('gpt2').to(device)
decoder_brain.eval()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [39]:
# 3. Stitch your prompt together using your RAG middleware glue string [prev]
rag_prompt = f"Based on the following reference protocol, answer the user question.\nContext: {fetched_context_document}\nQuestion: {user_question}\nAnswer:\n"

In [35]:
# # 4. AUTOREGRESSIVE WORD-BY-WORD WRITING LOOP [prev]
# max_new_tokens = 30
# print("\nGPT-2 Generator is typing...")

# for _ in range(max_new_tokens):
#   current_encoding = decoder_tokenizer(
#         rag_prompt,
#         padding='max_length',
#         max_length=256, # Expanded length to safely accommodate long fetched contexts [prev]
#         truncation=True,
#         return_tensors='pt'
#     )

#   interfrence_Dataset = SentimentDataset(encoding=current_encoding)

#   ids_tensor = interfrence_Dataset.encoding['input_ids'].to(device)
#   mask_tensor = interfrence_Dataset.encoding['attention_mask'].to(device)

#   with torch.no_grad():
#     with torch.amp.autocast('cuda'):

#       outputs = decoder_brain(input_ids=ids_tensor, attention_mask=mask_tensor)

#   next_token_logits = outputs.logits[:, -1, :]
#   # Pick the single word ID with the absolute highest score [prev]
#   next_token_id = torch.argmax(next_token_logits, dim=-1).item()

#   # Stop writing immediately if the model hits the End-of-Text boundary token [prev]
#   if next_token_id == decoder_tokenizer.eos_token_id:
#       break

#   # Translate that single number back into a clean English word string [prev]
#   next_word_string = decoder_tokenizer.decode([next_token_id])

#   # Glue the new word onto our prompt string so the model can read it on the next loop! [prev]
#   rag_prompt += next_word_string


GPT-2 Generator is typing...


/tmp/ipykernel_8568/4225976168.py:20: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with torch.amp.autocast('cuda'):


In [40]:
print("--- STEP 4: GENERATING VERIFIED RAG RESPONSE ---")

# 1. Tokenize the stitched prompt without padding to keep it clean [prev]
input_ids = decoder_tokenizer.encode(rag_prompt, return_tensors='pt').to(device)

# 2. Use the native production generator method to execute sampling search rules [prev]
with torch.no_grad():
    output_ids = decoder_brain.generate(
        input_ids,
        max_new_tokens=40,        # Number of words to write [prev]
        repetition_penalty=1.3,   # Hard ban on looping words like "The" [prev]
        no_repeat_ngram_size=2,   # Blocks duplicate phrase loops [prev]
        do_sample=True,           # Turns on smart word sampling [prev]
        top_k=50,                 # Filters out low-probability junk words [prev]
        temperature=0.7,          # Controls creativity/focus balance [prev]
        pad_token_id=decoder_tokenizer.eos_token_id
    )

# 3. Decode the final text output matrix array back into a human sentence [prev]
final_answer = decoder_tokenizer.decode(output_ids, skip_special_tokens=True)

print("\n--- FINAL RAG RESPONSE ---")
print(final_answer)

--- STEP 4: GENERATING VERIFIED RAG RESPONSE ---

--- FINAL RAG RESPONSE ---
["Based on the following reference protocol, answer the user question.\nContext: User Question: question about canceling a bloody purchase | Protocol Answer: For sure! I understand that you have some concerns regarding canceling your purchase. I'm here to help. Can you please provide me with the order number of your purchase so that I can assist you more effectively?\nQuestion: Hey, I need help canceling the purchase order I just made\nAnswer:\nHello (your name and address). Please see below for information associated terms if applicable in this case or any other similar situation which may be involved during transaction execution process... It is important not only do we"]


In [42]:
# 1. FIXED: Added [0] to extract the raw string out of the list container first!
clean_final_answer = final_answer[0].split("Answer:\n")[-1].strip()

# 2. Print the clean isolated answer block
print("\n--- CLEAN RAG ANSWER ---")
print(clean_final_answer)


--- CLEAN RAG ANSWER ---
Hello (your name and address). Please see below for information associated terms if applicable in this case or any other similar situation which may be involved during transaction execution process... It is important not only do we
